In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from transaction_analysis.paths import FRAUD_DATASET_DIR

transactions = pd.read_csv(FRAUD_DATASET_DIR / "raw" / "transactions_data.csv")

In [2]:
transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 13305915 entries, 0 to 13305914
Data columns (total 12 columns):
 #   Column          Dtype  
---  ------          -----  
 0   id              int64  
 1   date            str    
 2   client_id       int64  
 3   card_id         int64  
 4   amount          str    
 5   use_chip        str    
 6   merchant_id     int64  
 7   merchant_city   str    
 8   merchant_state  str    
 9   zip             float64
 10  mcc             int64  
 11  errors          str    
dtypes: float64(1), int64(5), str(6)
memory usage: 1.8 GB


In [3]:
from transaction_analysis.data.preprocess import zip_to_category

# # 1. Currency string ("$-77.00") -> Decimal32(9, 2): 4 bytes, exact, range ±$9,999,999.99
# #    pd.ArrowDtype keeps NA natively (no sentinel needed).
# amount_clean = transactions["amount"].replace(r"[\$,]", "", regex=True).str.strip()
# transactions["amount"] = pd.array(amount_clean, dtype=pd.ArrowDtype(pa.decimal32(9, 2)))

# # 2. Date string -> datetime64[us]; unparseable values become NaT.
# transactions["date"] = pd.to_datetime(transactions["date"], errors="coerce")

# # 3. int64 -> uintXX (all are non-negative ids); downcast picks the smallest unsigned dtype.
# for col in ["id", "client_id", "card_id", "merchant_id"]:
#     transactions[col] = pd.to_numeric(transactions[col], downcast="unsigned")

# # mcc is a 4-digit ISO 18245 code -> nullable UInt16 (preserves <NA>).
# transactions["mcc"] = transactions["mcc"].astype("UInt16")

# # 4. zip: float64 (with NaN) -> nullable Int64 -> string -> category; NaN flows through as <NA>.
# #    No zero-padding here; normalization/imputation happens in a later step.
# zip_int = transactions["zip"].astype("Int32")
# transactions["zip"] = zip_int.astype("string").astype("category")

# # 5. Low-cardinality strings -> category. Categorical dtype stores NaN as a real missing value
# #    (not a category), so unknowns stay distinguishable from any real label.
# for col in ["use_chip", "merchant_city", "merchant_state", "errors"]:
#     transactions[col] = transactions[col].astype("category")

transactions = zip_to_category(transactions, "zip")
transactions.info()
transactions.head()

<class 'pandas.DataFrame'>
RangeIndex: 13305915 entries, 0 to 13305914
Data columns (total 12 columns):
 #   Column          Dtype   
---  ------          -----   
 0   id              int64   
 1   date            str     
 2   client_id       int64   
 3   card_id         int64   
 4   amount          str     
 5   use_chip        str     
 6   merchant_id     int64   
 7   merchant_city   str     
 8   merchant_state  str     
 9   zip             category
 10  mcc             int64   
 11  errors          str     
dtypes: category(1), int64(5), str(6)
memory usage: 1.8 GB


,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
0,7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,,5499,NaN
1,7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,,5311,NaN
2,7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,,4829,NaN
3,7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,,4829,NaN
4,7475332,2010-01-01 00:06:00,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,,5813,NaN


In [4]:
transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 13305915 entries, 0 to 13305914
Data columns (total 12 columns):
 #   Column          Dtype                   
---  ------          -----                   
 0   id              uint32                  
 1   date            datetime64[us]          
 2   client_id       uint16                  
 3   card_id         uint16                  
 4   amount          decimal32(9, 2)[pyarrow]
 5   use_chip        category                
 6   merchant_id     uint32                  
 7   merchant_city   category                
 8   merchant_state  category                
 9   zip             category                
 10  mcc             UInt16                  
 11  errors          category                
dtypes: UInt16(1), category(5), datetime64[us](1), decimal32(9, 2)[pyarrow](1), uint16(2), uint32(2)
memory usage: 446.2 MB


In [ ]:
# transactions.to_parquet(FRAUD_DATASET_DIR / "cleaned" / "transactions_data.parquet", compression="zstd")